# EDA on ultimate marathon data

In [0]:
VOLUME_PATH = "/Volumes/marathos/default/raw/data"

spark.sql(f"LIST '{VOLUME_PATH}'").display()

In [0]:
%sql
SHOW SCHEMAS IN marathos;

In [0]:
DATA_PATH = "/Volumes/marathos/default/raw"

schema = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{DATA_PATH}/data/TWO_CENTURIES_OF_UM_RACES.csv")
    .schema
)

schema

In [0]:
df = spark.read.csv(f"{DATA_PATH}/data/TWO_CENTURIES_OF_UM_RACES.csv", header = True, inferSchema = True)

In [0]:
df.printSchema()

In [0]:
df.display()

## Specify schema for calculations

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, FloatType

schema = StructType([
    StructField("Year of event", IntegerType()),
    StructField("Event dates", StringType()),
    StructField("Event name", StringType()),
    StructField("Event distance/length", StringType()),
    StructField("Event number of finishers", IntegerType()),
    StructField("Athlete performance", StringType()),
    StructField("Athlete club", StringType()),
    StructField("Athlete country", StringType()),
    StructField("Athlete year of birth", DoubleType()),
    StructField("Athlete gender", StringType()),
    StructField("Athlete age category", StringType()),
    StructField("Athlete average speed", FloatType()),
    StructField("Athlete ID", IntegerType())
])

df_ultra_marathon_schema = spark.read.csv(f"{DATA_PATH}/data/TWO_CENTURIES_OF_UM_RACES.csv", header = True, schema = schema)
display(df_ultra_marathon_schema.take(5))

In [0]:
print(f"Number of rows {df_ultra_marathon_schema.count()}")
print(f"Number of columns {len(df_ultra_marathon_schema.columns)}")

In [0]:
df_ultra_marathon_schema.select(
    [
        column
        for column, type_ in df_ultra_marathon_schema.dtypes
        if type_ in ("int", "bigint", "double", "decimal")
    ]
).summary().display()

## Check null values

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df_ultra_marathon_schema.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df.columns]
)

null_counts = null_counts.collect()[0].asDict()
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

## Look at amount of unique events

In [0]:
from pyspark.sql import functions as sf
df_ultra_marathon_schema.select(sf.count_distinct("Event name")).show()

## Most represented countries

In [0]:
df_countries = df_ultra_marathon_schema.groupBy("Athlete country") \
                                       .count() \
                                       .withColumnRenamed("count", "times_represented") \
                                       .orderBy("times_represented", ascending = False)

df_countries.limit(10).display()

Databricks visualization. Run in Databricks to view.

In [0]:
# Amount of unique countries

df_ultra_marathon_schema.select(sf.count_distinct("Athlete country")).show()

## Ages in different ways

In [0]:
# Calculate ages for the specific year of the race

df_age = df_ultra_marathon_schema.withColumn("Age", col("Year of event") - col("Athlete year of birth").cast("float").cast("int"))
df_age.select("Age").summary().display()

In [0]:
# narrowing down ages for more accuracy 

df_age_filter = df_age.filter((col("Age") > 10) & (col("Age") < 90))
df_age_filter.select("Age").summary().display()

In [0]:
df_birth_years = df_ultra_marathon_schema.groupBy("Athlete year of birth") \
                                       .count() \
                                       .withColumnRenamed("count", "amount") \
                                       .orderBy("Athlete year of birth", ascending = True)

df_birth_years.limit(5).display()

In [0]:
# What birth year has most runners/for what area of birt years is ultra marathon popular

df_birth_years.display()

Databricks visualization. Run in Databricks to view.

In [0]:
# See what ages runs in what age category

df_age_filter.select("Age", "Athlete age category").limit(5).display()

In [0]:
from pyspark.sql.functions import length

# Check if any input in birth year is puin incorrectly (6 count cause the are doubles atm)
df_birth_year_length  = df_age.withColumn("char_count", length(col("Athlete year of birth")))

df_birth_year_length = df_birth_year_length.groupBy("char_count") \
                                            .count() \
                                            .orderBy("char_count") \
                                            .display()



## Information about athletes

In [0]:
# Unique amount of athletes

df_ultra_marathon_schema.select(sf.count_distinct("Athlete ID")).show()

In [0]:
# Amount of male and female athletes
# got help from LLM with .agg 

df_genders = df_ultra_marathon_schema.groupBy("Athlete gender") \
                                        .agg(sf.count_distinct("Athlete ID").alias("amount")) \
                                        .withColumnRenamed("count", "amount") \
                                        .orderBy("amount", ascending=False)

df_genders.display()                                      

Databricks visualization. Run in Databricks to view.

## Dates/years

In [0]:
# Amount of unique dates 
df_ultra_marathon_schema.select(sf.count_distinct("Event dates")).show()

In [0]:
# Amount of unique years

df_ultra_marathon_schema.select(sf.count_distinct("Year of event")).show()

## Information about "Event distance/length" and "Athlete performance"

In [0]:
df_club_athletes = df_ultra_marathon_schema.groupBy("Event distance/length") \
                                        .count() \
                                        .withColumnRenamed("count", "amount_of_athletes") \
                                        .orderBy("count", ascending = False)
                                            
df_club_athletes.limit(10).display()

In [0]:
df_performance_athletes = df_ultra_marathon_schema.groupBy("Athlete performance") \
                                        .count() \
                                        .withColumnRenamed("count", "amount_of_athletes") \
                                        .orderBy("count", ascending = True)
                                            
df_performance_athletes.limit(5).display()

## Clubs reprecented

In [0]:
# Amount of unique clubs

df_ultra_marathon_schema.select(sf.count_distinct("Athlete club")).show()

In [0]:
# Amount of represenatives from each club

df_club_athletes = df_ultra_marathon_schema.groupBy("Athlete club") \
                                        .count() \
                                        .withColumnRenamed("count", "amount_of_athletes") \
                                        .orderBy("count", ascending = False)
                                            
df_club_athletes.limit(5).display()

## EDA country reference dataset

- simulated by LLM

In [0]:
df_country = spark.read.csv(f"{DATA_PATH}/country_data/country_codes_reference.csv", header = True, inferSchema = True)

In [0]:
df_country.printSchema()

In [0]:
df_country.limit(5).display()

In [0]:
print(f"Number of rows {df_country.count()}")

## Compare countries in datasets

In [0]:
set(df_country.select("Country_Code").toPandas()["Country_Code"]).symmetric_difference(set(df_ultra_marathon_schema.select("Athlete country").toPandas()["Athlete country"]))

In [0]:
# See if ned and swe is included in dataset with big letters

df_country.select("Country_Code").filter("Country_Code IN ('NED', 'SWE')").display()

# Insights

### Amount of
- rows: 7 461 195 
- columns: 13
- columns with null values: 7
- unique events: 26 907

### Countries
- Most represented is USA
- unique amount of countries: 208 (ex. some have different spelling like sve and swe)
- the event holding country is in the same colummn as event name
- using the country_code_reference dataset -> can turn AUT to Austria
- "missing" countries in country reference dataset is there, they have big letters instead of small

### Ages
- Some birthyears are put in wrong (ex. one is 1193, 600 years before first data in dataset)
- the "Athlete age category" contains athletes older than the category (ex. M35 contains male athletes from 35 up to next age category)
- amount of nulls: 588 161

### Athletes
- amount of unique athletes: 1 641 168 
- Male athletes: 1 269 525
- Female athletes: 374 188
- Unknown/null:  20

### Dates/years
- unique amount of dates: 14 425
- unique amount of years: 146 (shows not all yeas have had a ultra marathon)
- year has a seperate column but is also included in date

### length/distance/performance/average speed
- if the event is a distance then performance is time (and the other way around)
- time can be measured both in d (day) and h (hour)
- distance can be measured both in km (kilometers) and mi (miles, 10 kilometers)
- average speed is measured in km/h

### Clubs
- Unique amount of clubs reprecented: 552 189 
- amount of nulls: 2 826 373 

### Data types
- Event dates -> date
- Athlete average speed -> float/double